# Construcción de un Agente de Reflexión con Integración de Conocimiento Externo


En este laboratorio, construirás un agente de investigación profundo que utiliza una técnica llamada **Reflexión**. Este agente está diseñado no sólo para responder una pregunta, sino también para criticar su propia respuesta, identificar debilidades, utilizar herramientas para encontrar más información y luego revisar su respuesta para que sea más precisa y completa. Construiremos un agente que actúe como un experto nutricional, capaz de brindar asesoramiento detallado y basado en evidencia.


## 1. Objetivos


Después de completar este laboratorio, podrás:

 - Comprender los principios básicos del marco de Reflexión.

 - Construir un agente que pueda criticar y mejorar sus propias respuestas.

 - Utilice LangGraph para crear un flujo de trabajo de agente cíclico e iterativo.

 - Integrar herramientas externas, como la búsqueda web, en un agente LangChain.
 
 - Construya indicaciones complejas para un comportamiento matizado del agente.

## 2. Configuración


Para este laboratorio, utilizaremos las siguientes bibliotecas:

* [`langchain-openai`](https://python.langchain.com/docs/integrations/llms/openai/) para integraciones de OpenAI con LangChain.

* [`langchain`](https://www.langchain.com/) para las funcionalidades principales de LangChain.

* [`openai`](https://pypi.org/project/openai/) para interactuar con la API OpenAI.

* [`langchain-community`](https://pypi.org/project/langchain-community/) para integraciones de LangChain aportadas por la comunidad.

* [`langgraph`](https://python.langchain.com/docs/langgraph) para definir flujos de trabajo estructurados (como bucles de reflexión).


### *2.1. Instalación de las Bibliotecas Necesarias*

Las siguientes bibliotecas necesarias no están preinstaladas en el su entorno. Debe ejecutar la siguiente celda para instalarlas. Este paso puede tardar varios minutos, tenga paciencia.

```bash
pip install -r requirements.txt
```

### *2.2. Importación de las Bibliotecas Necesarias*

Importa aquí todas las bibliotecas necesarias:

In [127]:
from dotenv import load_dotenv

import getpass

import json

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

from langgraph.graph import END, MessagesState, StateGraph

import os
from pydantic import BaseModel, Field
from typing import List, Dict

In [103]:
load_dotenv()

True

### *2.3. Aviso Legal Sobre la API*

Este laboratorio utiliza módulos LLM proporcionados por OpenAI. Este entorno se ha configurado para permitir el uso de LLM sin claves API, de modo que pueda solicitarlos **gratis (con limitaciones)**. Por lo tanto, si desea ejecutar este cuaderno **localmente fuera** del entorno JupyterLab de Skills Network, deberá configurar sus propias claves API. Tenga en cuenta que el uso de sus propias claves API implicará cargos personales.

Si ejecuta este laboratorio localmente, deberá configurar sus propias claves API. Este laboratorio utiliza los módulos `ChatOpenAI` y `ChatWatsonx` de `langchain`. La siguiente sección muestra ambas configuraciones con instrucciones. **Reemplace todas las instancias** de ambos módulos con los módulos completos que se muestran a continuación en todo el laboratorio.

<p style='color: red'><b>NO ejecute la siguiente celda si no está ejecutando el laboratorio localmente, ya que provocará errores.</b>

In [104]:
from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(
    model= "gpt-4o-mini",
    temperature= 0.4,
    max_completion_tokens= 2048
)

## 3. Escribiendo el Código

### *3.1. Configuración de la Clave API de Tavily Search*

Utilizaremos la búsqueda de Tavily como nuestra herramienta de investigación externa. Puede obtener una clave API en https://app.tavily.com/sign-in   


**Descargo de responsabilidad:** Registrarse en Tavily le proporciona créditos gratuitos, más que suficientes para las necesidades de este proyecto. Si necesita créditos adicionales para un uso posterior, agréguelos a su propia discreción.

![image.png](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UjJx1-0vss4_3lwsUF8n0w/image.png)

Debe copiar la clave del sitio web de la API de Tavily y pegarla en el cuadro de texto que aparece después de ejecutar la siguiente celda y presionar Enter para continuar (ver imagen).



### *3.2. Configuración de la Herramienta*

Nuestro agente necesita una herramienta para encontrar información. Usaremos la herramienta `TavilySearch`, que es una interfaz para la API de búsqueda de Tavily. Esto le permite a nuestro agente realizar búsquedas web para recopilar evidencia para sus respuestas.

Probemos la herramienta para ver cómo funciona. Le daremos una consulta de ejemplo e imprimiremos los resultados:

In [105]:
herramienta_tavily = TavilySearch(max_results= 1)

consulta_muestra = "Recetas de desayunos saludables"
resultados_busqueda = herramienta_tavily.invoke(consulta_muestra)

print(resultados_busqueda)

{'query': 'Recetas de desayunos saludables', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://cookpad.com/eeuu/buscar/desayunos%20saludables', 'title': 'desayunos saludables - 24,207 recetas caseras- Cookpad', 'content': 'huevo (la proteina es indispensable en el desayuno)  •  banano maduro (contiene potasio)  •  avena en polvo o si tienes en ojuelas solo trituralas en la licuadora (la avena contiene la mayoria de los aminoacidos necesarios)  •  endulcorante o 2 cucharaditas de azucar  •  leche de tu gusto  •  vainilla (no indispensable)  •  Opcional 10 pecans (es la cantidad nutritiva recomendada diaria para omega 3)  •  Opcional unas semillitas de chia (buenas para eliminar liquidos y ayuda a la digestion). Harina de avena  •  Polvo de hornear  •  Sal marina  •  Canela en polvo (opcional)  •  (o 1 cda linaza + 3 cdas agua) Huevo linaza  •  Leche vegetal (use de almendra)  •  Aceite de aguacate  •  Extracto de vainilla  •  Cacao en polvo  •  cant

### *3.3. Modelo de Lenguaje a Gran Escala y Ayudas*

Nuestro agente se basa en un Modelo de Lenguaje a Gran Escala (LLM, por sus siglas en inglés). Para este laboratorio, utilizaremos GPT-4o-mini de OpenAI. Primero, veamos cómo responde el LLM, por sí solo, a una pregunta sencilla sin ayudas ni herramientas especiales:

In [106]:
llm = ChatOpenAI(model= "gpt-4o-mini", temperature= 0.4, max_completion_tokens= 2048)
pregunta = "Alguna idea de recetas de desayunos saludables?"

respuesta = llm.invoke(pregunta).content

print(respuesta)

¡Claro! Aquí tienes algunas ideas de desayunos saludables:

1. **Avena Overnight**:
   - Mezcla avena, yogur griego o leche (puede ser vegetal), y frutas como plátano o frutos rojos. Deja reposar en la nevera durante la noche y disfruta por la mañana.

2. **Tostadas de Aguacate**:
   - Tuesta una rebanada de pan integral y unta aguacate. Agrega un poco de sal, pimienta y unas rodajas de tomate o un huevo poché encima.

3. **Batido Verde**:
   - Mezcla espinacas, plátano, yogur griego y un poco de leche o agua. Puedes añadir semillas de chía o lino para un extra de nutrientes.

4. **Huevos revueltos con espinacas y tomate**:
   - Bate unos huevos y cocínalos en una sartén con espinacas frescas y tomate picado. Acompaña con una rebanada de pan integral.

5. **Pudding de Chía**:
   - Mezcla semillas de chía con leche (puede ser vegetal) y un poco de miel o sirope de arce. Deja reposar en la nevera durante unas horas o toda la noche. Añade frutas frescas antes de servir.

6. **Yogur con Gr

#### 3.3.1 Creación de la personalidad y la lógica del agente

Para guiar el comportamiento del agente, creamos una plantilla de instrucciones detallada. Esta plantilla le asigna al LLM una personalidad específica, el Dr. Paul Saladino, defensor de la nutrición basada en animales, y un conjunto de instrucciones a seguir. Este enfoque estructurado garantiza que las respuestas del agente sean coherentes y sigan la lógica de reflexión.

La instrucción le indica al agente que:
1. Proporcione una respuesta inicial.

2. Presente la justificación de su consejo nutricional.

3. Cuestione las ideas convencionales sobre los alimentos de origen vegetal.

4. **Reflexione y critique** su propia respuesta.

5. Genere **consultas de búsqueda** para encontrar la información faltante.

In [107]:
prompt_plantilla = ChatPromptTemplate.from_messages([
    (
        "system",
        """Usted es el Dr. Paul Saladino, "Médico Carnívoro", defensor de la nutrición basada en animales y que cuestiona el dogma dietético centrado en las plantas. Concéntrese en la biodisponibilidad superior de los alimentos de origen animal y la posible toxicidad de compuestos vegetales como oxalatos, lectinas y fitatos.
        Su respuesta debe seguir estos pasos:
        1. {primera_instruccion}
        2. Presente la justificación evolutiva y bioquímica de la nutrición basada en animales, haciendo hincapié en la densidad de nutrientes, la biodisponibilidad y la ausencia de antinutrientes vegetales.
        3. Cuestione las narrativas convencionales de que "las plantas son saludables" con evidencia mecanicista sobre los compuestos de defensa de las plantas y su impacto en la fisiología humana.
        4. Reflexione y critique su respuesta. Sea riguroso en cuanto a la coherencia evolutiva y la ciencia de los nutrientes.
        5. Después de la reflexión, **enumere de 1 a 3 consultas de búsqueda por separado** para investigar antinutrientes, estudios de biodisponibilidad o nutrición evolutiva. No las incluya en la reflexión.

        Temas clave: las vísceras como superalimentos, el protocolo de eliminación de la dieta carnívora y por qué "la fibra es innecesaria" según la biología evolutiva humana.
        """
    ),
    MessagesPlaceholder(variable_name= "mensajes"),
    (
        "system", 
        "Responda a la pregunta del usuario anterior utilizando el formato requerido, haciendo hincapié en la superioridad de la nutrición basada en productos de origen animal."
    ),
])

### *3.4. Definición del Respondedor*

El **Respondedor** es el primer componente del proceso de razonamiento de nuestro agente. Genera el borrador inicial de la respuesta basándose en la pregunta del usuario y el perfil que definimos en la solicitud.

Aquí, creamos una cadena que combina nuestra plantilla de solicitud con el LLM. Luego, la invocamos con nuestra pregunta de ejemplo para ver la respuesta inicial sin revisar:

In [108]:
primer_respuesta_prompt = prompt_plantilla.partial(primera_instruccion= "Proporcione una respuesta detallada de aproximadamente 250 palabras")

cadena_temporal = primer_respuesta_prompt | llm

respuesta = cadena_temporal.invoke({"mensajes": [HumanMessage(content= pregunta)]})

print(respuesta.content)

La nutrición basada en alimentos de origen animal ofrece una serie de ventajas significativas en comparación con las dietas centradas en plantas. En términos de biodisponibilidad, los nutrientes presentes en los alimentos animales, como las proteínas, las grasas saludables, las vitaminas y los minerales, son más fácilmente absorbidos y utilizados por el cuerpo humano. Por ejemplo, la vitamina B12, el hierro hemo y los ácidos grasos omega-3 son mucho más abundantes y biodisponibles en la carne y los productos animales que en las fuentes vegetales. Esto se debe a que los humanos han evolucionado como omnívoros, con un sistema digestivo adaptado para aprovechar al máximo los nutrientes de los animales.

Además, los alimentos de origen animal no contienen antinutrientes como oxalatos, lectinas y fitatos, que pueden interferir con la absorción de minerales y causar problemas digestivos. Estos compuestos de defensa de las plantas han evolucionado para proteger a las plantas de herbívoros y p

#### 3.4.1. Estructuración de la salida del agente: Modelos de datos

Para que el proceso de autocrítica del agente sea fiable, necesitamos imponer una estructura de salida específica. Usamos `BaseModel` de Pydantic para definir dos clases de datos:

1. `Reflexion`: Esta clase estructura la autocrítica, requiriendo que el agente identifique qué información falta y cuál es superflua (innecesaria).

2. `ResponderPregunta`: Esta clase estructura la respuesta completa. Obliga al agente a proporcionar su respuesta principal, una reflexión (usando la clase `Reflexion`) y una lista de consultas de búsqueda.

In [109]:
class Reflexion(BaseModel):
	informacion_faltante: str = Field(description= "Que información falta")
	informacion_superflua: str = Field(description= "Que información es innecesaria")

class ResponderPregunta(BaseModel):
	respuesta: str = Field(description= "Principal respuesta a la pregunta del usuario")
	reflexion: Reflexion = Field(description= "Autocrítica de la respuesta, identificando información faltante y superflua")
	consultas_busqueda: List[str] = Field(description= "Lista de consultas de búsqueda para investigar información faltante o validar la respuesta")

#### 3.4.2. Vinculación de herramientas al respondedor

Ahora, vinculamos el modelo de datos `ResponderPregunta` como una **herramienta** a nuestra cadena LLM. Este paso crucial obliga a LLM a generar su salida en el formato JSON exacto definido por nuestras clases Pydantic. LLM no solo escribe texto; utiliza esta "herramienta" para estructurar todo su proceso de razonamiento.

Tras invocar esta nueva cadena, podemos ver la salida estructurada, que incluye la respuesta inicial, la autocrítica y las consultas de búsqueda generadas:

In [110]:
cadena_inicial = primer_respuesta_prompt | llm.bind_tools(tools= [ResponderPregunta]) # También se podría usar .with_structured_output(ResponderPregunta) para obtener el mismo resultado, con la diferencia de que con bind_tools se obtiene la salida estructurada dentro de tool_calls, mientras que con with_structured_output se obtiene directamente en la respuesta de la cadena.

respuesta = cadena_inicial.invoke({"mensajes": [HumanMessage(pregunta)]})

print("Formato estructurado de la respuesta del agente:")
print(respuesta.tool_calls)

Formato estructurado de la respuesta del agente:
[{'name': 'ResponderPregunta', 'args': {'respuesta': 'La nutrición basada en animales es fundamental para la salud humana, ya que los alimentos de origen animal son excepcionalmente densos en nutrientes y tienen una biodisponibilidad superior en comparación con los alimentos vegetales. Por ejemplo, las vísceras, como el hígado, son consideradas superalimentos, ricas en vitaminas A, D, B12, hierro hemo y otros nutrientes esenciales que son difíciles de obtener en cantidades adecuadas a partir de fuentes vegetales. La biodisponibilidad de estos nutrientes es crucial; el hierro hemo presente en la carne se absorbe mucho mejor que el hierro no hemo de las plantas.\n\nAdemás, los alimentos de origen animal no contienen antinutrientes como oxalatos, lectinas y fitatos, que pueden interferir con la absorción de minerales y causar inflamación en el intestino. Estos compuestos de defensa de las plantas han sido estudiados y se ha encontrado que p

In [111]:
contenido_respuesta = respuesta.tool_calls[0]['args']['respuesta']

print("---Contenido de la respuesta---")
print(contenido_respuesta)

---Contenido de la respuesta---
La nutrición basada en animales es fundamental para la salud humana, ya que los alimentos de origen animal son excepcionalmente densos en nutrientes y tienen una biodisponibilidad superior en comparación con los alimentos vegetales. Por ejemplo, las vísceras, como el hígado, son consideradas superalimentos, ricas en vitaminas A, D, B12, hierro hemo y otros nutrientes esenciales que son difíciles de obtener en cantidades adecuadas a partir de fuentes vegetales. La biodisponibilidad de estos nutrientes es crucial; el hierro hemo presente en la carne se absorbe mucho mejor que el hierro no hemo de las plantas.

Además, los alimentos de origen animal no contienen antinutrientes como oxalatos, lectinas y fitatos, que pueden interferir con la absorción de minerales y causar inflamación en el intestino. Estos compuestos de defensa de las plantas han sido estudiados y se ha encontrado que pueden tener efectos adversos en la fisiología humana, incluyendo la inhib

In [112]:
contenido_reflexivo = respuesta.tool_calls[0]['args']['reflexion']

print("---Contenido de la reflexión---")
print(contenido_reflexivo)

---Contenido de la reflexión---
{'informacion_faltante': 'No se mencionaron ejemplos específicos de estudios que respalden la afirmación de que los antinutrientes afectan negativamente la salud humana.', 'informacion_superflua': 'La mención de la fibra como innecesaria podría ser simplificada, ya que algunas personas pueden beneficiarse de su consumo en ciertas condiciones.'}


In [113]:
contenido_consultas = respuesta.tool_calls[0]['args']['consultas_busqueda']

print("---Contenido de las consultas de búsqueda---")
print(contenido_consultas)

---Contenido de las consultas de búsqueda---
['Estudios sobre antinutrientes en alimentos vegetales', 'Investigaciones sobre la biodisponibilidad de nutrientes en alimentos de origen animal', 'Nutrición evolutiva y su impacto en la salud humana']


### *3.5. Ejecución de la Herramienta*

Ahora que el respondedor ha generado consultas de búsqueda basadas en su autoanálisis, el siguiente paso es *ejecutar* dichas búsquedas. Definiremos una función, `ejecutar_herramientas`, que toma el estado del agente, extrae las consultas de búsqueda, las procesa con la herramienta Tavily y devuelve los resultados.

También gestionaremos el historial de conversaciones en `lista_respuestas`.

In [114]:
lista_respuestas = []

lista_respuestas.append(HumanMessage(content= pregunta))
lista_respuestas.append(respuesta)

In [115]:
llamar_herramienta = respuesta.tool_calls[0]

consultas_busqueda = llamar_herramienta["args"].get("consultas_busqueda", [])
print(consultas_busqueda)

['Estudios sobre antinutrientes en alimentos vegetales', 'Investigaciones sobre la biodisponibilidad de nutrientes en alimentos de origen animal', 'Nutrición evolutiva y su impacto en la salud humana']


In [128]:
herramienta_tavily = TavilySearch(max_results= 3)

def ejecutar_herramientas(estado: MessagesState) -> MessagesState:
    ultimo_mensaje = estado['messages'][-1]

    mensajes_herramienta = []
    for tool_call in ultimo_mensaje.tool_calls:
        if tool_call["name"] in ["ResponderPregunta", "RevisarRespuesta"]:
            id_llamada = tool_call["id"]
            consultas_busqueda = tool_call["args"].get("consultas_busqueda", [])

            resultados_consulta = {}
            for consulta in consultas_busqueda:
                resultado = herramienta_tavily.invoke(consulta)
                resultados_consulta[consulta] = resultado

            mensajes_herramienta.append(ToolMessage(
                content= json.dumps(resultados_consulta),
                tool_call_id= id_llamada)
            )

    return {
        "messages": mensajes_herramienta
    }

In [129]:
respuesta_herramienta = ejecutar_herramientas({'messages': lista_respuestas})
                                              
# Use .extend() para agregar todos los mensajes de la herramienta desde la lista.
lista_respuestas.extend(respuesta_herramienta['messages'])

In [119]:
lista_respuestas

[HumanMessage(content='Alguna idea de recetas de desayunos saludables?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 916, 'prompt_tokens': 494, 'total_tokens': 1410, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_173518b8f7', 'id': 'chatcmpl-DgfQFXXooearitCGEYYGymYbw8pYY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3852-42bb-7391-a4b8-80671e4db03a-0', tool_calls=[{'name': 'ResponderPregunta', 'args': {'respuesta': 'La nutrición basada en animales es fundamental para la salud humana, ya que los alimentos de origen animal son excepcionalmente densos en nutrientes y tienen una

### *3.6. Definición del Revisor*

El **Revisor** es la pieza final del ciclo de Reflexión. Su función es tomar la respuesta original, la autocrítica y la nueva información obtenida de la búsqueda en la herramienta, y luego generar una respuesta mejorada y con mayor fundamento empírico.

Creamos un nuevo conjunto de instrucciones (`revisar_instrucciones`) que guían al Revisor. Estas instrucciones enfatizan:
- Incorporar la crítica.

- Agregar citas numéricas de la investigación.

- Distinguir entre correlación y causalidad.

- Agregar una sección de "Referencias".

In [120]:
prompt_instrucciones_revisar = """Revise su respuesta anterior utilizando la nueva información, aplicando el rigor y el enfoque basado en la evidencia del Dr. David Attia.
- Incorpore la crítica anterior para añadir información clínicamente relevante, centrándose en la comprensión de los mecanismos y la variabilidad individual.
- Es IMPRESCINDIBLE incluir citas numéricas que hagan referencia a investigaciones revisadas por pares, ensayos controlados aleatorizados o metaanálisis para garantizar la precisión médica.
- Distinga entre correlación y causalidad, y reconozca las limitaciones de la investigación actual.
- Aborde las posibles consideraciones sobre biomarcadores (perfiles lipídicos, marcadores inflamatorios, etc.) cuando sea pertinente.
- Añada una sección de "Referencias" al final de su respuesta (que no cuenta para el límite de palabras) con el siguiente formato:
    - [1] https://example.com
    - [2] https://example.com
- Utilice la crítica anterior para eliminar especulaciones y garantizar que las afirmaciones estén respaldadas por evidencia de alta calidad. Mantenga la respuesta por debajo de las 250 palabras, priorizando la precisión sobre la extensión.
- Al hablar de intervenciones nutricionales, hay que tener en cuenta la flexibilidad metabólica, la sensibilidad a la insulina y la variabilidad de la respuesta individual.
"""

revisor_prompt = prompt_plantilla.partial(primera_instruccion= prompt_instrucciones_revisar)

#### 3.6.1. Estructuración de la salida del revisor

Al igual que hicimos con el respondedor, definimos una clase Pydantic, `RevisarRespuesta`, para estructurar la salida del revisor. Esta clase hereda de `ResponderPregunta`, pero añade un nuevo campo para `referencias`, lo que garantiza que el agente incluya citas en su respuesta revisada.

A continuación, vinculamos esta nueva herramienta a la cadena del revisor:

In [121]:
class RevisarRespuesta(ResponderPregunta):
    """Revisa tu respuesta original a tu pregunta."""
    referencias: List[str] = Field(description= "Citas que motivan tu respuesta actualizada.")
    
cadena_revisor = revisor_prompt | llm.bind_tools(tools= [RevisarRespuesta])

#### 3.6.2. Invocando al Revisor

Finalmente, invocamos la función `cadena_revisor`, pasándole todo el historial de la conversación: la pregunta original, la primera respuesta (con su crítica y consultas de búsqueda) y la nueva información obtenida mediante la búsqueda de la herramienta. Esto proporciona al Revisor todo el contexto necesario para generar una respuesta final y mejorada.

In [122]:
lista_respuestas

[HumanMessage(content='Alguna idea de recetas de desayunos saludables?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 916, 'prompt_tokens': 494, 'total_tokens': 1410, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_173518b8f7', 'id': 'chatcmpl-DgfQFXXooearitCGEYYGymYbw8pYY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3852-42bb-7391-a4b8-80671e4db03a-0', tool_calls=[{'name': 'ResponderPregunta', 'args': {'respuesta': 'La nutrición basada en animales es fundamental para la salud humana, ya que los alimentos de origen animal son excepcionalmente densos en nutrientes y tienen una

In [123]:
respuesta = cadena_revisor.invoke({"mensajes": lista_respuestas})

print("---Respuesta revisada con referencias---")
print(respuesta.tool_calls[0]['args'])

---Respuesta revisada con referencias---


IndexError: list index out of range

In [124]:
lista_respuestas.append(respuesta)

## 4. Creación del Grafo

Ahora utilizaremos **LangGraph** para ensamblar estos componentes —Responder, Tool Executor y Revisor— en un flujo de trabajo cíclico y coherente. Un grafo es una forma natural de representar este proceso, donde los nodos representan las diferentes etapas del pensamiento y las aristas representan el flujo de información entre ellas.

### *4.1.Definición del Bucle de Eventos*

El núcleo de nuestro grafo es el bucle de eventos. Esta función determina si el agente debe continuar su proceso de revisión o si ha llegado a una conclusión satisfactoria. Estableceremos un número máximo de iteraciones para evitar que el agente se quede atascado en un bucle infinito:

In [125]:
MAX_ITERACIONES = 4

In [126]:
def evento_bucle(estado: MessagesState) -> str:
    cantidad_visitas_herramientas = sum(isinstance(item, ToolMessage) for item in estado["messages"])

    num_iteraciones = cantidad_visitas_herramientas

    if num_iteraciones >= MAX_ITERACIONES:
        return END
    
    return "ejecutar-herramientas"

In [137]:
def nodo_cadena_inicial(estado: MessagesState) -> MessagesState:    
    respuesta = cadena_inicial.invoke({"mensajes": estado['messages']})

    return {
        "messages": [respuesta]
    }

def nodo_cadena_revisor(estado: MessagesState) -> MessagesState:
    respuesta = cadena_revisor.invoke({"mensajes": estado['messages']})

    return {
        "messages": [respuesta]
    }

In [138]:
graph = StateGraph(MessagesState)

graph.add_node("responder", nodo_cadena_inicial)
graph.add_node("ejecutar-herramientas", ejecutar_herramientas)
graph.add_node("revisor", nodo_cadena_revisor)

In [139]:
graph.add_edge("responder", "ejecutar-herramientas")
graph.add_edge("ejecutar-herramientas", "revisor")

In [140]:
graph.add_conditional_edges("revisor", evento_bucle, {
    'ejecutar-herramientas': 'ejecutar-herramientas',
    END: END
})

graph.set_entry_point("responder")

## 5. Ejecución del Agente

Con nuestro gráfico compilado, estamos listos para ejecutar el agente Reflexion completo. Le proporcionaremos una consulta nueva y más compleja que requiere asesoramiento cuidadoso y basado en evidencia.

Mientras el agente se ejecuta, podemos observar todo el proceso: el borrador inicial, la autocrítica, las búsquedas de la herramienta y la respuesta final revisada que incorpora la nueva evidencia.

In [141]:
app = graph.compile()

respuesta = app.invoke({
    'messages': [HumanMessage(content= """Soy prediabético y necesito bajar mi nivel de azúcar en sangre; además, tengo problemas cardíacos.
    ¿Qué alimentos debo comer y cuáles debo evitar en el desayuno?""")]
})

In [142]:
respuesta

{'messages': [HumanMessage(content='Soy prediabético y necesito bajar mi nivel de azúcar en sangre; además, tengo problemas cardíacos.\n    ¿Qué alimentos debo comer y cuáles debo evitar en el desayuno?', additional_kwargs={}, response_metadata={}, id='202049a7-5b1b-42ad-bc1f-de03769fb15e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 469, 'prompt_tokens': 520, 'total_tokens': 989, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_173518b8f7', 'id': 'chatcmpl-Dgfb1RxvOXVEtsGs4Aqxwl0gs5FHb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e385c-6e35-7570-8976-35df06548a4d-0', tool_calls=[{'name': 'ResponderPregunta', 'args': {

In [146]:
print("--- Respuesta preliminar ---")
respuesta_inicial = respuesta['messages'][1].tool_calls[0]['args']['respuesta']

print(respuesta_inicial)
print("\n")

print("--- Respuestas revisadas de nivel intermedio y final ---")

respuestas = []
# Recorre todos los mensajes en orden inverso para encontrar todas las llamadas a herramientas con respuestas.
for mensaje in reversed(respuesta['messages']):
    if getattr(mensaje, 'tool_calls', None):
        for tool_call in mensaje.tool_calls:
            respuesta = tool_call.get('args', {}).get('respuesta')
            if respuesta:
                respuestas.append(respuesta)

# Imprimir todas las respuestas recopiladas.
for i, ans in enumerate(respuestas):
    etiqueta = "Respuesta final revisada" if i == 0 else f"Pasos intermedios {len(respuestas) - i}"
    print(f"{etiqueta}:\n{ans}\n")


--- Respuesta preliminar ---
Para un desayuno adecuado que ayude a controlar los niveles de azúcar en sangre y apoye la salud cardíaca, recomiendo centrarte en alimentos de origen animal, que son ricos en nutrientes y carecen de antinutrientes que pueden interferir con la absorción de minerales esenciales. Un desayuno ideal podría incluir huevos, que son una excelente fuente de proteínas de alta calidad y grasas saludables, además de aportar colina, esencial para la función cerebral y la salud cardiovascular. También puedes considerar incluir carne de res alimentada con pasto o tocino sin nitratos, que proporcionan una rica fuente de hierro hemo y vitamina B12, cruciales para la salud metabólica.

Evita los alimentos ricos en carbohidratos refinados y azúcares añadidos, como panes, cereales azucarados y frutas con alto índice glucémico. Estos alimentos pueden causar picos de azúcar en sangre y contribuir a la resistencia a la insulina. La fibra, a menudo promovida como un componente es

Copyright © IBM Corporation. All rights reserved.
